# Notebook 21 — Certification Boundary Sweep

Certification boundary sweep for `quantum-hardware-company/control-stack`.

This notebook varies adversarial attack strength and identifies the first point where each policy crosses the 45° CGCS phase-lock threshold.


In [ ]:
# ============================================================
# Notebook 21 — Certification Boundary Sweep
# quantum-hardware-company / control-stack
# ============================================================

import json, math, os, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_ID = "21"
SLUG = "certification_boundary_sweep"
TITLE = "Certification Boundary Sweep"
SEED = 9423
THRESHOLD = 1 / np.sqrt(2)

ROOT = Path(".")
FIG_DIR = ROOT / "figures"
RESULTS_DIR = ROOT / "results"
DOCS_DIR = ROOT / "docs"
for d in [FIG_DIR, RESULTS_DIR, DOCS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(SEED)

print(f"{NOTEBOOK_ID}_{SLUG}")
print(f"45° threshold = {THRESHOLD:.6f}")


## 1. Synthetic calibration stack

Use the same control-stack abstraction as Notebooks 16–20: drift traces, measurement noise, burst windows, and mismatch windows.


In [ ]:
# ============================================================
# Synthetic calibration stack
# ============================================================

N_BLOCKS = 88
blocks = np.arange(N_BLOCKS)

# Baseline drift fields: smooth + structured calibration drift.
omega_true = (
    0.035 * np.sin(2*np.pi*blocks/22)
    + 0.018 * np.sin(2*np.pi*blocks/9)
    + 0.00055 * blocks
)
beta_true = (
    0.012 * np.sin(2*np.pi*blocks/18)
    + 0.00095 * blocks
)

# Repeated disturbance windows used throughout control-stack notebooks.
NOISE_WINDOWS = [(31, 42), (48, 56), (62, 75)]
MISMATCH_WINDOWS = [(58, 75), (77, 86)]

def in_windows(i, windows):
    return any(a <= i < b for a, b in windows)

def shade_windows(ax):
    first_noise, first_mismatch = True, True
    for a, b in NOISE_WINDOWS:
        ax.axvspan(a, b, alpha=0.16, label="noise burst" if first_noise else None)
        first_noise = False
    for a, b in MISMATCH_WINDOWS:
        ax.axvspan(a, b, alpha=0.08, label="model mismatch" if first_mismatch else None)
        first_mismatch = False

def simulate_measurements(attack_strength=0.0, family="phase_shift_window"):
    """Return measured omega/beta with attack scaled by attack_strength."""
    rng_local = np.random.default_rng(SEED + int(round(1000 * attack_strength)))
    noise_o = rng_local.normal(0, 0.006, N_BLOCKS)
    noise_b = rng_local.normal(0, 0.004, N_BLOCKS)

    # Larger measurement variance inside noise windows.
    for i in range(N_BLOCKS):
        if in_windows(i, NOISE_WINDOWS):
            noise_o[i] += rng_local.normal(0, 0.020)
            noise_b[i] += rng_local.normal(0, 0.014)

    omega_meas = omega_true + noise_o
    beta_meas = beta_true + noise_b

    # Optimizer-like compact attack: default is a phase-shift window.
    start, duration = 20, 14
    end = min(N_BLOCKS, start + duration)
    window = np.zeros(N_BLOCKS)
    window[start:end] = np.hanning(duration) if duration > 1 else 1.0

    if family == "phase_shift_window":
        omega_meas += attack_strength * 0.18 * window
        beta_meas += attack_strength * 0.11 * window
    elif family == "correlated_burst":
        omega_meas += attack_strength * 0.13 * window
        beta_meas += attack_strength * 0.13 * window
    elif family == "delayed_measurement":
        delay = int(2 + round(4 * attack_strength))
        omega_meas[start:end] = np.roll(omega_meas, delay)[start:end]
        beta_meas[start:end] = np.roll(beta_meas, delay)[start:end]
        omega_meas += attack_strength * 0.05 * window
        beta_meas += attack_strength * 0.03 * window
    else:
        raise ValueError(f"unknown family: {family}")

    return omega_meas, beta_meas

def target_response(omega, beta):
    """Ideal phase-aligned response probability."""
    t = np.linspace(0, 10, 240)
    phase = 0.45 * omega + 0.35 * beta
    response = 0.5 + 0.5*np.sin((1.55 + phase) * t[:, None])**2
    return t, response.T

def cosine_similarity(a, b, eps=1e-12):
    aa = np.asarray(a)
    bb = np.asarray(b)
    return float(np.dot(aa, bb) / (np.linalg.norm(aa)*np.linalg.norm(bb) + eps))


## 2. Controller policies

Compare baseline, moving average, Kalman-style policies, MPC-like constrained control, CGCS invariance control, and oracle.


In [ ]:
# ============================================================
# Controller policies
# ============================================================

def moving_average(x, k=5):
    y = np.zeros_like(x)
    for i in range(len(x)):
        y[i] = np.mean(x[max(0, i-k+1):i+1])
    return y

def kalman_1d(z, q=0.00018, r=0.004):
    x = np.zeros_like(z)
    p = 1.0
    x[0] = z[0]
    for i in range(1, len(z)):
        xp = x[i-1]
        pp = p + q
        k = pp / (pp + r)
        x[i] = xp + k * (z[i] - xp)
        p = (1-k) * pp
    return x

def robust_gated(z, gate=0.075, q=0.00012, r=0.003):
    x = np.zeros_like(z)
    p = 1.0
    x[0] = z[0]
    rejects = []
    for i in range(1, len(z)):
        xp = x[i-1]
        innovation = z[i] - xp
        if abs(innovation) > gate:
            z_eff = xp + np.sign(innovation) * gate
            rejects.append(i)
        else:
            z_eff = z[i]
        pp = p + q
        k = pp / (pp + r)
        x[i] = xp + k * (z_eff - xp)
        p = (1-k) * pp
    return x, rejects

def joint_kalman(omega_z, beta_z):
    # Lightweight joint filter: independent Kalman plus coupled correction.
    o = kalman_1d(omega_z, q=0.00020, r=0.0045)
    b = kalman_1d(beta_z, q=0.00016, r=0.0035)
    coupling = 0.18 * moving_average(o - np.mean(o), 7)
    b = b + 0.25 * coupling
    return o, b

def constrained_mpc_like(omega_z, beta_z):
    o = moving_average(omega_z, 4)
    b = moving_average(beta_z, 6)
    # Smooth control-like command limiting.
    o = np.clip(o, omega_true - 0.08, omega_true + 0.08)
    b = np.clip(b, beta_true - 0.06, beta_true + 0.06)
    return o, b

def cgcs_invariance_control(omega_z, beta_z, update_radius=0.075):
    # Start from a joint estimate, then constrain updates by projected CGCS radius.
    o0, b0 = joint_kalman(omega_z, beta_z)
    o = np.zeros_like(o0)
    b = np.zeros_like(b0)
    o[0], b[0] = o0[0], b0[0]
    rejects = []
    for i in range(1, len(o0)):
        do = o0[i] - o[i-1]
        db = b0[i] - b[i-1]
        step = math.sqrt(do*do + db*db)
        if step > update_radius:
            scale = update_radius / (step + 1e-12)
            do *= scale
            db *= scale
            rejects.append(i)
        o[i] = o[i-1] + do
        b[i] = b[i-1] + db
    # Tiny blend toward smooth model to avoid high-frequency ringing.
    o = 0.88 * o + 0.12 * moving_average(omega_z, 7)
    b = 0.88 * b + 0.12 * moving_average(beta_z, 7)
    return o, b, rejects

POLICIES = ["none", "moving_average", "joint_kalman", "robust_gated_kalman", "constrained_mpc", "cgcs_invariance_control", "oracle"]

def run_policy(policy, omega_meas, beta_meas):
    if policy == "none":
        return omega_meas.copy(), beta_meas.copy(), []
    if policy == "moving_average":
        return moving_average(omega_meas, 5), moving_average(beta_meas, 5), []
    if policy == "joint_kalman":
        o, b = joint_kalman(omega_meas, beta_meas)
        return o, b, []
    if policy == "robust_gated_kalman":
        o, ro = robust_gated(omega_meas, gate=0.075)
        b, rb = robust_gated(beta_meas, gate=0.055)
        return o, b, sorted(set(ro + rb))
    if policy == "constrained_mpc":
        o, b = constrained_mpc_like(omega_meas, beta_meas)
        return o, b, []
    if policy == "cgcs_invariance_control":
        o, b, rej = cgcs_invariance_control(omega_meas, beta_meas, update_radius=0.075)
        return o, b, rej
    if policy == "oracle":
        return omega_true.copy(), beta_true.copy(), []
    raise ValueError(policy)

def evaluate_policy(policy, attack_strength, family="phase_shift_window"):
    om, bm = simulate_measurements(attack_strength, family)
    ohat, bhat, rejects = run_policy(policy, om, bm)

    t, target = target_response(omega_true, beta_true)
    _, pred = target_response(ohat, bhat)

    cos = np.array([cosine_similarity(pred[i], target[i]) for i in range(N_BLOCKS)])
    rmse = np.sqrt(np.mean((pred - target)**2, axis=1))
    margin = cos - THRESHOLD

    return {
        "policy": policy,
        "attack_strength": attack_strength,
        "family": family,
        "omega_meas": om,
        "beta_meas": bm,
        "omega_hat": ohat,
        "beta_hat": bhat,
        "cos": cos,
        "rmse": rmse,
        "margin": margin,
        "rejects": rejects,
        "min_margin": float(np.min(margin)),
        "mean_rmse": float(np.mean(rmse)),
        "p95_rmse": float(np.percentile(rmse, 95)),
        "blocks_below_threshold": int(np.sum(margin < 0)),
    }


## 3. Sweep adversarial strength

For each policy and attack strength, compute minimum CGCS margin, RMSE, p95 RMSE, and blocks below threshold.


In [ ]:
# ============================================================
# Certification boundary sweep
# ============================================================

strengths = np.round(np.linspace(0.0, 1.8, 19), 3)
family = "phase_shift_window"

rows = []
cache = {}
for s in strengths:
    for p in POLICIES:
        res = evaluate_policy(p, s, family)
        cache[(p, s)] = res
        rows.append({
            "policy": p,
            "attack_strength": s,
            "min_margin": res["min_margin"],
            "mean_rmse": res["mean_rmse"],
            "p95_rmse": res["p95_rmse"],
            "blocks_below_threshold": res["blocks_below_threshold"],
        })

sweep_df = pd.DataFrame(rows)
sweep_path = RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_boundary_sweep.csv"
sweep_df.to_csv(sweep_path, index=False)
sweep_df.head()


## 4. Critical boundary detection

Critical attack strength is the first strength where `minimum margin < 0`.


In [ ]:
# ============================================================
# Critical strength detection
# ============================================================

critical_rows = []
for p in POLICIES:
    sdf = sweep_df[sweep_df["policy"] == p].sort_values("attack_strength")
    below = sdf[sdf["min_margin"] < 0]
    if len(below) == 0:
        critical = np.inf
        label = "certified-through-sweep"
    else:
        critical = float(below.iloc[0]["attack_strength"])
        label = "boundary-crossed"
    critical_rows.append({
        "policy": p,
        "critical_attack_strength": critical,
        "boundary_label": label,
        "final_min_margin": float(sdf.iloc[-1]["min_margin"]),
        "max_blocks_below_threshold": int(sdf["blocks_below_threshold"].max()),
        "mean_rmse_at_final_strength": float(sdf.iloc[-1]["mean_rmse"]),
    })

critical_df = pd.DataFrame(critical_rows).sort_values(
    ["critical_attack_strength", "final_min_margin"],
    ascending=[False, False]
).reset_index(drop=True)

critical_path = RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_critical_strength.csv"
critical_df.to_csv(critical_path, index=False)
critical_df


## 5. Figures


In [ ]:
# ============================================================
# Figure helpers
# ============================================================

def savefig(name):
    path = FIG_DIR / f"{NOTEBOOK_ID}_{SLUG}_{name}.png"
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    print(f"saved: {path}")
    return path

figure_names = []


In [ ]:
# Figure 1 — critical attack strength

plot_df = critical_df.copy()
plot_df["critical_plot"] = plot_df["critical_attack_strength"].replace(np.inf, strengths[-1] + 0.2)

plt.figure(figsize=(11, 5.8))
plt.bar(plot_df["policy"], plot_df["critical_plot"])
plt.axhline(strengths[-1], linestyle="--", label="max swept strength")
plt.ylabel("critical attack strength")
plt.title("Certification boundary sweep: critical attack strength")
plt.xticks(rotation=28, ha="right")
plt.legend()
savefig("critical_attack_strength")
figure_names.append("critical_attack_strength")


In [ ]:
# Figure 2 — min CGCS margin curves

plt.figure(figsize=(12, 6))
for p in POLICIES:
    sdf = sweep_df[sweep_df["policy"] == p]
    plt.plot(sdf["attack_strength"], sdf["min_margin"], marker="o", label=p)
plt.axhline(0, linestyle="--", label="threshold margin = 0")
plt.xlabel("attack strength multiplier")
plt.ylabel("minimum CGCS margin")
plt.title("Certification boundary sweep: minimum margin vs attack strength")
plt.legend(ncol=2)
savefig("min_margin_curves")
figure_names.append("min_margin_curves")


In [ ]:
# Figure 3 — phase diagram: stable/unstable

pol_order = list(critical_df["policy"])
phase_mat = []
for p in pol_order:
    sdf = sweep_df[sweep_df["policy"] == p].sort_values("attack_strength")
    phase_mat.append((sdf["min_margin"].values < 0).astype(int))
phase_mat = np.array(phase_mat)

plt.figure(figsize=(12, 5.8))
plt.imshow(phase_mat, aspect="auto", interpolation="nearest")
plt.yticks(np.arange(len(pol_order)), pol_order)
plt.xticks(np.arange(len(strengths)), [str(s) for s in strengths], rotation=45)
plt.xlabel("attack strength multiplier")
plt.ylabel("policy")
plt.title("Certification boundary sweep: phase diagram (failure = margin < 0)")
plt.colorbar(label="failure")
savefig("phase_diagram")
figure_names.append("phase_diagram")


In [ ]:
# Figure 4 — RMSE vs attack strength

plt.figure(figsize=(12, 6))
for p in POLICIES:
    sdf = sweep_df[sweep_df["policy"] == p]
    plt.plot(sdf["attack_strength"], sdf["mean_rmse"], marker="o", label=p)
plt.xlabel("attack strength multiplier")
plt.ylabel("mean response RMSE")
plt.title("Certification boundary sweep: accuracy degradation")
plt.legend(ncol=2)
savefig("rmse_curves")
figure_names.append("rmse_curves")


In [ ]:
# Figure 5 — blocks below threshold vs attack strength

plt.figure(figsize=(12, 6))
for p in POLICIES:
    sdf = sweep_df[sweep_df["policy"] == p]
    plt.plot(sdf["attack_strength"], sdf["blocks_below_threshold"], marker="o", label=p)
plt.xlabel("attack strength multiplier")
plt.ylabel("blocks below 45° threshold")
plt.title("Certification boundary sweep: threshold violations")
plt.legend(ncol=2)
savefig("threshold_violation_curves")
figure_names.append("threshold_violation_curves")


In [ ]:
# Figure 6 — boundary scatter: accuracy vs final adversarial margin

final_df = sweep_df[sweep_df["attack_strength"] == strengths[-1]].copy()
plt.figure(figsize=(9, 6))
plt.scatter(final_df["mean_rmse"], final_df["min_margin"], s=90)
for _, r in final_df.iterrows():
    plt.annotate(r["policy"], (r["mean_rmse"], r["min_margin"]), xytext=(6, 4), textcoords="offset points")
plt.axhline(0, linestyle="--")
plt.xlabel("mean RMSE at max attack strength")
plt.ylabel("minimum CGCS margin at max attack strength")
plt.title("Certification boundary sweep: accuracy vs final stability margin")
savefig("accuracy_vs_final_margin")
figure_names.append("accuracy_vs_final_margin")


In [ ]:
# Figure 7 — worst boundary waveform at first failure for best non-oracle policy

non_oracle = critical_df[critical_df["policy"] != "oracle"].copy()
selected_policy = non_oracle.iloc[0]["policy"]
selected_strength = non_oracle.iloc[0]["critical_attack_strength"]
if not np.isfinite(selected_strength):
    selected_strength = strengths[-1]

res = cache[(selected_policy, float(selected_strength))]
worst_block = int(np.argmin(res["margin"]))
t, target = target_response(omega_true, beta_true)

plt.figure(figsize=(11, 5.8))
plt.plot(t, target[worst_block], linewidth=2.5, label="target")
for p in ["none", "moving_average", "joint_kalman", "robust_gated_kalman", "constrained_mpc", "cgcs_invariance_control", "oracle"]:
    r = cache[(p, float(selected_strength))]
    _, pred = target_response(r["omega_hat"], r["beta_hat"])
    plt.plot(t, pred[worst_block], label=p)
plt.xlabel("time / pulse duration")
plt.ylabel("excited-state probability")
plt.title(f"Certification boundary sweep: waveform at boundary — {selected_policy}, block {worst_block}, strength {selected_strength}")
plt.legend(ncol=2)
savefig("boundary_waveform")
figure_names.append("boundary_waveform")


In [ ]:
# Figure 8 — disturbance and estimator traces at boundary

om, bm = simulate_measurements(float(selected_strength), family)
plt.figure(figsize=(12, 6))
shade_windows(plt.gca())
plt.plot(blocks, omega_true, linewidth=2, label="true Ω drift")
plt.plot(blocks, om, marker="o", alpha=0.65, label="measured Ω")
for p in ["joint_kalman", "robust_gated_kalman", "constrained_mpc", "cgcs_invariance_control"]:
    r = cache[(p, float(selected_strength))]
    plt.plot(blocks, r["omega_hat"], label=p)
plt.xlabel("calibration block")
plt.ylabel("Ω drift estimate / command")
plt.title(f"Certification boundary sweep: Ω trace at strength {selected_strength}")
plt.legend(ncol=2)
savefig("omega_trace_at_boundary")
figure_names.append("omega_trace_at_boundary")


## 6. Report, manifest, README snippet, zip export


In [ ]:
# ============================================================
# Write markdown report and manifest
# ============================================================

best_policy = critical_df.iloc[0]["policy"]
best_boundary = critical_df.iloc[0]["critical_attack_strength"]
best_boundary_text = "beyond max swept strength" if not np.isfinite(best_boundary) else f"{best_boundary:.3f}"

md_lines = []
md_lines.append(f"# Notebook {NOTEBOOK_ID} — {TITLE}")
md_lines.append("")
md_lines.append("## Summary")
md_lines.append("")
md_lines.append(
    "Notebook 21 sweeps adversarial attack strength to locate certification boundaries. "
    "Instead of testing one optimized attack, it maps when each controller first crosses the 45° CGCS phase-lock threshold."
)
md_lines.append("")
md_lines.append("## Certification boundary result")
md_lines.append("")
md_lines.append(f"- Attack family: `{family}`")
md_lines.append(f"- 45° threshold: `{THRESHOLD:.6f}`")
md_lines.append(f"- Best policy by boundary strength: `{best_policy}`")
md_lines.append(f"- Best boundary: `{best_boundary_text}`")
md_lines.append("")
md_lines.append("## Critical attack strength table")
md_lines.append("")
md_lines.append(critical_df.to_markdown(index=False))
md_lines.append("")
md_lines.append("## Interpretation")
md_lines.append("")
md_lines.append(
    "A controller is treated as certified through a sweep range when its minimum CGCS margin stays non-negative across the attack strengths tested. "
    "Boundary crossing occurs at the first attack strength where minimum margin drops below zero. "
    "This converts Notebook 20's scorecard into a stability phase diagram: accuracy degradation and phase-lock loss are measured separately."
)
md_lines.append("")
md_lines.append("## Figures")
md_lines.append("")
for name in figure_names:
    md_lines.append(f"![{name}](../figures/{NOTEBOOK_ID}_{SLUG}_{name}.png)")
    md_lines.append("")

md_path = DOCS_DIR / f"{NOTEBOOK_ID}_{SLUG}.md"
md_path.write_text("\n".join(md_lines))
print(f"saved markdown: {md_path}")

manifest = {
    "notebook": f"{NOTEBOOK_ID}_{SLUG}",
    "title": TITLE,
    "threshold": float(THRESHOLD),
    "seed": SEED,
    "attack_family": family,
    "strengths": [float(x) for x in strengths],
    "best_policy": str(best_policy),
    "figures": [str(FIG_DIR / f"{NOTEBOOK_ID}_{SLUG}_{name}.png") for name in figure_names],
    "results": [
        str(sweep_path),
        str(critical_path),
    ],
    "docs": [str(md_path)],
}
manifest_path = RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))
print(f"saved manifest: {manifest_path}")


In [ ]:
# ============================================================
# README snippet
# ============================================================

readme_snippet = f"""
### Notebook {NOTEBOOK_ID}: {TITLE}

Colab:

`https://colab.research.google.com/github/thinkthoughts/quantum-hardware-company/blob/main/control-stack/{NOTEBOOK_ID}_{SLUG}.ipynb`

Purpose:
- sweep adversarial attack strength
- estimate first phase-lock boundary crossing
- separate RMSE degradation from CGCS margin collapse
- export phase diagram, critical-strength table, figures, report, and manifest

Primary result:
- certification boundary is measured by first strength where `min_margin < 0`
- policies that remain non-negative across the sweep are certified through the tested range
""".strip()

snippet_path = DOCS_DIR / f"{NOTEBOOK_ID}_{SLUG}_README_snippet.md"
snippet_path.write_text(readme_snippet)
print(readme_snippet)


In [ ]:
# ============================================================
# Zip export bundle
# ============================================================

zip_name = f"{NOTEBOOK_ID}_{SLUG}_outputs.zip"
with zipfile.ZipFile(zip_name, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in manifest["figures"] + manifest["results"] + manifest["docs"] + [str(snippet_path)]:
        pth = Path(p)
        if pth.exists():
            zf.write(pth, pth.as_posix())
print(f"created {zip_name}")

# Optional Colab download. Safe to run only inside Colab.
try:
    from google.colab import files
    files.download(zip_name)
except Exception as e:
    print("Colab download skipped:", repr(e))
